# LFW·SurvFace·RFW-Custom 통합 Quick/Full 실행기

이 노트북은 기존 데이터셋별 노트북을 대체하지 않고, 동일한 canonical `research/` 단계 함수를 한 곳에서 순차 호출하는 상위 실행기입니다.

- `quick`: LFW 10%, SurvFace 2%, RFW-Custom 10%의 identity-aware·role/group-preserving 표본을 사용합니다.
- `full`: 선택 데이터셋의 전체 표본 100%를 사용합니다.
- `arc`, `ada`, `mag`, `edge` 중 한 pretrained checkpoint를 선택합니다. 네 모델 비교는 모델별 독립 run을 반복하며 비교 단위는 checkpoint입니다.
- `DATASET_IDS`는 `lfw`, `survface`, `rfw_custom` 중 원하는 조합을 받습니다.
- 각 open-set run은 Origin, PCA direct/reconstruction, PQ reconstruction/ADC, Grad-CAM, saliency×compression, compact artifact를 유지합니다.
- FPIR 10%와 1%는 동일 검색 점수를 재사용해 `frozen_origin`과 `recalibrated_compressed` 정책을 모두 산출합니다.
- RFW-Custom은 비공식 1:N DIR/FPIR 경로입니다. 선택적 RFW-Official 1:1 TAR/FAR/EER는 별도 supplementary artifact로만 생성합니다.
- 실제 장시간 실험은 `EXECUTE=True`와 실행 확인 값이 모두 설정된 경우에만 시작됩니다.
- Faiss IVF-PQ, pgvector IVFFlat, ANN sweep, BalancedFace, uncertainty/defer 실험은 이번 구현 범위에서 유예합니다.


## 1. 사용자가 조절하는 변수

`DATASET_IDS`, `RUN_TIER`, `MODEL_NAME`을 선택합니다. `quick`의 데이터셋별 비율은 `QUICK_DATA_FRACTIONS`에 기록되고 `full`은 항상 100%입니다.

`MODEL_PROFILE_BY_NAME`과 `MODEL_WEIGHT_PATHS`는 같은 학습 데이터·아키텍처의 checkpoint여야 합니다. `START_NEW_RUN=True`는 동일 plan의 완료 run이 있어도 독립 재실험을 의도할 때만 사용합니다. RFW-Official 평가는 `RUN_RFW_VERIFICATION`으로 별도 활성화하며 RFW-Custom open-set 결과와 합치지 않습니다.


In [1]:
from __future__ import annotations

DATASET_IDS = ("rfw_custom",)
DATASET_ID = DATASET_IDS[0]  # 기존 단일-dataset 점검 코드 호환 별칭
RUN_TIER = "full"  # "quick" 또는 "full"
QUICK_DATA_FRACTIONS = {
    "lfw": 1.0,
    "survface": 1.0,
    "rfw_custom": 1.0,
}
TARGET_FPIRS = (0.10, 0.01)  # canonical config와 report가 검증
SEED = 42

MODEL_NAME = "ada"  # "arc", "ada", "mag", "edge" 중 하나
MODEL_PROFILE_BY_NAME = {
    "arc": "arcface_ms1mv3_r100",
    "ada": "adaface_ms1mv3_r100",
    "mag": "magface_ms1mv2_iresnet100",
    "edge": "edgeface_webface12m_xs_gamma_06",
}
MODEL_WEIGHT_PATHS = {
    "arc": "models/arcface/ms1mv3_r100_backbone.pth",
    "ada": "models/adaface/adaface_ir101_ms1mv3.ckpt",
    "mag": "models/magface/magface_ms1mv2.pth",
    "edge": "models/edgeface/edgeface_xs_gamma_06.pt",
}
# AdaFace MS1MV2 bridge를 쓸 때는 profile/checkpoint를 함께 변경합니다.
MODEL_SMOKE_DEVICE = "cuda"
ARTIFACT_STORAGE_MODE = "results_only"

EXECUTE = True
ACKNOWLEDGE_LOCAL_EXECUTION = True
START_NEW_RUN = False
COMPLETED_RUN_OVERRIDES = {
    # "lfw": "runs/...",
    # "survface": "runs/...",
    # "rfw_custom": (
    #     "runs/rfw_custom_20260810/"
    #     "20260810-R001-49ce74ce_step4_rfw_custom_"
    #     "edgeface-a348c305af33c223b337"
    # ),
}
RUN_SEARCH_SPACE_REFRESH = True
RUN_SURVFACE_FAITHFULNESS = True
RUN_FINAL_REPORT = True
WRITE_FINAL_REPORT = True

# RFW-Official 1:1 supplementary 평가. RFW-Custom 1:N과 별도입니다.
RUN_RFW_VERIFICATION = False
RFW_ORIGIN_ARTIFACT_DIR = "results/rfw_step7/origin_embeddings/{model_uid}"
# 동일 물리 RFW population에서 fit한 codec은 Official external-transfer 근거가 아닙니다.
RFW_CODEC_SOURCE_DATASETS = ("lfw", "survface")
RFW_SELECTED_CODEC_FAMILIES = ("pca", "pq")
RFW_SELECTED_CODEC_PROFILES = None
RFW_ALLOW_ORIGIN_ONLY = False
RFW_BOOTSTRAP_REPEATS = 2000
RFW_REUSE_COMPLETED = True


## 2. 프로젝트와 공통 runner 로드

현재 작업 디렉터리의 상위에서 저장소 루트를 찾습니다. 아래 셀은 실험을 시작하거나 가중치를 로드하지 않습니다.


In [2]:
from pathlib import Path
from pprint import pprint
import sys


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("C:/ronbun 프로젝트 루트를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.experiments.pipeline_runner import (
    FULL_DATA_FRACTION,
    build_common_experiment_plan,
    inspect_common_experiment_plan,
    prepare_common_model_checkpoint,
    reuse_completed_run_for_plan,
    run_common_step4_experiment,
)
from research.runtime import ProgressReporter
from research.experiments import (
    evaluate_rfw_frozen_codecs,
    frozen_codec_specs_from_completed_run,
    rfw_frozen_codec_evaluation_uid,
)
from scripts.run_integrated_postprocessing import (
    postprocess_completed_run,
    run_cross_dataset_report_notebook,
    verify_cross_model_faithfulness,
)

if not DATASET_IDS or not set(DATASET_IDS).issubset({"lfw", "survface", "rfw_custom"}):
    raise ValueError(f"지원하지 않는 DATASET_IDS: {DATASET_IDS!r}")
if len(set(DATASET_IDS)) != len(DATASET_IDS):
    raise ValueError(f"DATASET_IDS에 중복이 있습니다: {DATASET_IDS!r}")

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"MODEL_NAME={MODEL_NAME}")
print(f"DATASET_IDS={DATASET_IDS}")
print(f"QUICK_DATA_FRACTIONS={QUICK_DATA_FRACTIONS}")
print(f"TARGET_FPIRS={TARGET_FPIRS}")
print(f"FULL_DATA_FRACTION={FULL_DATA_FRACTION}")


PROJECT_ROOT=C:\ronbun
MODEL_NAME=ada
DATASET_IDS=('rfw_custom',)
QUICK_DATA_FRACTIONS={'lfw': 1.0, 'survface': 1.0, 'rfw_custom': 1.0}
TARGET_FPIRS=(0.1, 0.01)
FULL_DATA_FRACTION=1.0


## 3. 선택 모델과 가중치 고정

선택한 별칭에서 profile과 가중치 경로를 가져와 checkpoint SHA-256, 전처리, target layer를 하나의 `model_uid`로 등록합니다. 등록 정보가 같으면 재사용합니다. 기존 smoke 검증 결과가 있으면 재사용하고, 없으면 최대 8장으로 forward/target-layer smoke test를 수행합니다. 전체 데이터 실험은 시작하지 않습니다.


In [3]:
if MODEL_NAME not in MODEL_PROFILE_BY_NAME or MODEL_NAME not in MODEL_WEIGHT_PATHS:
    raise ValueError("MODEL_NAME은 'arc', 'ada', 'mag', 'edge' 중 하나여야 합니다.")

MODEL_PROFILE = MODEL_PROFILE_BY_NAME[MODEL_NAME]
MODEL_WEIGHT_PATH = PROJECT_ROOT / MODEL_WEIGHT_PATHS[MODEL_NAME]

MODEL_PREPARATION = prepare_common_model_checkpoint(
    project_root=PROJECT_ROOT,
    model_name=MODEL_NAME,
    model_profile=MODEL_PROFILE,
    checkpoint_path=MODEL_WEIGHT_PATH,
    run_smoke_validation=True,
    smoke_device=MODEL_SMOKE_DEVICE,
    seed=SEED,
)
pprint(MODEL_PREPARATION.as_dict(), sort_dicts=False)


{'model_name': 'ada',
 'model_profile': 'adaface_ms1mv3_r100',
 'family': 'adaface',
 'checkpoint_path': 'C:\\ronbun\\models\\adaface\\adaface_ir101_ms1mv3.ckpt',
 'checkpoint_sha256': 'f0bdc94fd6aa8fec7cd82c428288991ebeca2510f71949e68b89aee016fbaf82',
 'model_uid': 'adaface-4df25b75e065b0b9ed43',
 'model_spec_path': 'C:\\ronbun\\runs\\step2\\model_registry\\adaface-4df25b75e065b0b9ed43.json',
 'smoke_validation_status': 'reused_validated',
 'smoke_validation_path': 'C:\\ronbun\\runs\\step2\\model_validation\\adaface-4df25b75e065b0b9ed43\\smoke_summary.json'}


## 4. 결정적 실행 plan 생성

manifest를 읽어 실제 선택 예정 행 수와 role/split 분포를 계산합니다. 같은 manifest hash·seed·tier라면 같은 identity 집합을 선택합니다. 이 셀도 DB나 run을 변경하지 않습니다.


In [4]:
PLANS = {
    dataset_id: build_common_experiment_plan(
        project_root=PROJECT_ROOT,
        dataset_id=dataset_id,
        run_tier=RUN_TIER,
        seed=SEED,
        model_name=MODEL_NAME,
        model_profile=MODEL_PREPARATION.model_profile,
        model_uid=MODEL_PREPARATION.model_uid,
        model_checkpoint_path=MODEL_PREPARATION.checkpoint_path,
        quick_data_fractions=QUICK_DATA_FRACTIONS,
        artifact_storage_mode=ARTIFACT_STORAGE_MODE,
    )
    for dataset_id in DATASET_IDS
}
# 4 models × 3 datasets 통합 표를 만들 때만 12개 완료 run을 명시합니다.
# 자동 latest 선택은 하지 않으며 model/dataset/run lineage가 모두 검증됩니다.
CROSS_MODEL_RUN_MATRIX = {
    # "arcface": {"lfw": "runs/...", "survface": "runs/...", "rfw_custom": "runs/..."},
    # "adaface": {"lfw": "runs/...", "survface": "runs/...", "rfw_custom": "runs/..."},
    # "magface": {"lfw": "runs/...", "survface": "runs/...", "rfw_custom": "runs/..."},
    # "edgeface": {"lfw": "runs/...", "survface": "runs/...", "rfw_custom": "runs/..."},
}
for dataset_id, plan in PLANS.items():
    observed_targets = tuple(
        float(value)
        for value in plan.effective_step4_config["evaluation"]["reported_target_fpirs"]
    )
    if observed_targets != TARGET_FPIRS:
        raise ValueError(
            f"{dataset_id}: FPIR target contract mismatch: {observed_targets}"
        )
PLAN = PLANS[DATASET_ID]  # 기존 단일-plan 점검 코드와의 호환용 별칭
for dataset_id, plan in PLANS.items():
    print(f"\n[{dataset_id}] plan")
    pprint(plan.as_dict(), sort_dicts=False)



[rfw_custom] plan
{'plan_id': '356b55b7b06eddf2',
 'pipeline_id': 'common_step4_gradcam_v1',
 'dataset_id': 'rfw_custom',
 'run_tier': 'full',
 'data_fraction': 1.0,
 'quick_data_fractions': {'lfw': 1.0, 'survface': 1.0, 'rfw_custom': 1.0},
 'quick_fraction_override': True,
 'seed': 42,
 'model_name': 'ada',
 'model_profile': 'adaface_ms1mv3_r100',
 'model_uid': 'adaface-4df25b75e065b0b9ed43',
 'model_checkpoint_path': 'C:\\ronbun\\models\\adaface\\adaface_ir101_ms1mv3.ckpt',
 'evaluation_contract_id': 'face_search_evaluation_v1',
 'evaluation_contract_path': 'C:\\ronbun\\configs\\experiments\\evaluation_contract_v1.yaml',
 'evaluation_contract_sha256': '75bbaa5f00101f14eda4b83d265c86d36a4469684a61aa138f06f2120e004420',
 'base_step4_config_path': 'C:\\ronbun\\configs\\experiments\\step2_pytorch_gradcam.yaml',
 'source_rows': 40520,
 'selected_source_rows': 40520,
 'source_manifest_sha256': {'data/raw/RFW/images/test.tar.gz': '2ec1875599d01888431be83fe51ed9614188a026539fb970653081ca0fb

## 5. 로컬 preflight

등록 checkpoint, CUDA, ONNX Runtime provider, canonical aligned/landmark bundle, 로컬 Git source 상태를 읽기 전용으로 검사합니다. GitHub나 원격 CI는 사용하지 않습니다. quick은 dirty source를 허용하되 commit과 diff hash를 plan에 고정하고, full은 clean local commit을 요구합니다. `ready_to_execute_pipeline=False`이면 아래 실행 셀의 오류에 표시되는 실패 항목을 먼저 해결합니다.


In [5]:
PREFLIGHTS = {
    dataset_id: inspect_common_experiment_plan(plan)
    for dataset_id, plan in PLANS.items()
}
PREFLIGHT = PREFLIGHTS[DATASET_ID]  # 기존 단일-plan 점검 코드와의 호환용 별칭
for dataset_id, preflight in PREFLIGHTS.items():
    print(f"\n[{dataset_id}] preflight")
    pprint(preflight, sort_dicts=False)



[rfw_custom] preflight
{'plan': {'plan_id': '356b55b7b06eddf2',
          'pipeline_id': 'common_step4_gradcam_v1',
          'dataset_id': 'rfw_custom',
          'run_tier': 'full',
          'data_fraction': 1.0,
          'quick_data_fractions': {'lfw': 1.0,
                                   'survface': 1.0,
                                   'rfw_custom': 1.0},
          'quick_fraction_override': True,
          'seed': 42,
          'model_name': 'ada',
          'model_profile': 'adaface_ms1mv3_r100',
          'model_uid': 'adaface-4df25b75e065b0b9ed43',
          'model_checkpoint_path': 'C:\\ronbun\\models\\adaface\\adaface_ir101_ms1mv3.ckpt',
          'evaluation_contract_id': 'face_search_evaluation_v1',
          'evaluation_contract_path': 'C:\\ronbun\\configs\\experiments\\evaluation_contract_v1.yaml',
          'evaluation_contract_sha256': '75bbaa5f00101f14eda4b83d265c86d36a4469684a61aa138f06f2120e004420',
          'base_step4_config_path': 'C:\\ronbun\\configs\\e

## 6. 사용자 승인 후 순차 실행 또는 재개

`EXECUTE=True`, `ACKNOWLEDGE_LOCAL_EXECUTION=True`일 때만 실제 실험을 시작합니다. 실행 전에 위 plan과 preflight에서 데이터셋·fraction·model_uid·checkpoint 경로를 확인하십시오.

완료된 phase는 건너뛰고, 실패하거나 아직 실행하지 않은 phase부터 이어갑니다. 장시간 loop 로그는 약 10% 경계에서만 출력됩니다. 같은 plan의 완료 run이 있으면 자동으로 새 run을 만들지 않습니다.


In [6]:
EXECUTION_RESULTS = {}
POSTPROCESS_RESULTS = {}
FINAL_REPORT_RESULT = {"status": "not_started"}
RFW_RESULT = {"status": "not_started"}

if EXECUTE:
    if ACKNOWLEDGE_LOCAL_EXECUTION is not True:
        raise RuntimeError(
            "실제 실행 전 ACKNOWLEDGE_LOCAL_EXECUTION=True가 필요합니다."
        )
    for dataset_id, preflight in PREFLIGHTS.items():
        if preflight["ready_to_execute_pipeline"]:
            continue
        CHECKS = preflight["readiness"]["checks"]
        FAILED_CHECKS = {
            key: CHECKS.get(key)
            for key in (
                "git_policy_satisfied",
                "source_snapshot_matches",
                "cuda_available",
                "required_onnx_provider_available",
                "model_spec_verified",
            )
            if CHECKS.get(key) is not True
        }
        raise RuntimeError(f"preflight 실패: {FAILED_CHECKS}")

    for dataset_id, plan in PLANS.items():
        progress = ProgressReporter(
            f"{dataset_id}/{RUN_TIER}/{MODEL_NAME}",
            heartbeat_seconds=None,
            milestone_percent=10,
        )
        override = COMPLETED_RUN_OVERRIDES.get(dataset_id)
        execution = (
            reuse_completed_run_for_plan(plan, PROJECT_ROOT / override)
            if override
            else run_common_step4_experiment(
                plan,
                execution_acknowledged=True,
                start_new_run=START_NEW_RUN,
                progress=progress.callback(
                    key_prefix=f"{dataset_id}:{RUN_TIER}:"
                ),
            )
        )
        EXECUTION_RESULTS[dataset_id] = execution
        if execution.get("status") not in {"completed", "already_completed"}:
            raise RuntimeError(f"{dataset_id}: 완료 run을 확보하지 못했습니다: {execution}")

    # 모든 dataset run을 먼저 완료한 뒤 파생 artifact를 만든다. 첫 dataset의
    # results가 다음 dataset의 source preflight에 영향을 주지 않게 한다.
    for dataset_id, execution in EXECUTION_RESULTS.items():
        POSTPROCESS_RESULTS[dataset_id] = postprocess_completed_run(
            execution["run_dir"],
            refresh_search_spaces=RUN_SEARCH_SPACE_REFRESH,
            derive_survface_faithfulness=RUN_SURVFACE_FAITHFULNESS,
        )

    if RUN_RFW_VERIFICATION:
        origin_dir = PROJECT_ROOT / RFW_ORIGIN_ARTIFACT_DIR.format(
            model_uid=MODEL_PREPARATION.model_uid
        )
        codec_specs = []
        for source_dataset in RFW_CODEC_SOURCE_DATASETS:
            if source_dataset not in EXECUTION_RESULTS:
                continue
            codec_specs.extend(
                frozen_codec_specs_from_completed_run(
                    EXECUTION_RESULTS[source_dataset]["run_dir"],
                    expected_model_uid=MODEL_PREPARATION.model_uid,
                    families=RFW_SELECTED_CODEC_FAMILIES,
                    profile_names=RFW_SELECTED_CODEC_PROFILES,
                )
            )
        codec_specs = tuple(codec_specs)
        if not codec_specs and not RFW_ALLOW_ORIGIN_ONLY:
            raise RuntimeError(
                "RFW verification requires frozen codecs from an explicitly "
                "selected completed run, or RFW_ALLOW_ORIGIN_ONLY=True."
            )
        rfw_uid = rfw_frozen_codec_evaluation_uid(
            origin_artifact_dir=origin_dir,
            codec_specs=codec_specs,
            strict_official=True,
            bootstrap_seed=SEED,
            bootstrap_repeats=RFW_BOOTSTRAP_REPEATS,
        )
        rfw_output_dir = (
            PROJECT_ROOT / "results/rfw_step7/frozen_codec_evaluation"
            / MODEL_PREPARATION.model_uid / rfw_uid
        )
        rfw_evaluation = evaluate_rfw_frozen_codecs(
            origin_artifact_dir=origin_dir,
            codec_specs=codec_specs,
            output_dir=rfw_output_dir,
            strict_official=True,
            bootstrap_seed=SEED,
            bootstrap_repeats=RFW_BOOTSTRAP_REPEATS,
            reuse_completed=RFW_REUSE_COMPLETED,
        )
        RFW_RESULT = {
            "status": "completed",
            "evaluation_uid": rfw_uid,
            "output_dir": str(rfw_evaluation.root),
            "profile_rows": len(rfw_evaluation.profile_summary),
            "codec_count": len(codec_specs),
        }

    if RUN_FINAL_REPORT:
        survface_execution = EXECUTION_RESULTS.get("survface")
        cross_model = verify_cross_model_faithfulness(
            PROJECT_ROOT,
            expected_survface_run_id=(
                survface_execution["run_id"] if survface_execution else None
            ),
            expected_model_uid=(
                MODEL_PREPARATION.model_uid if survface_execution else None
            ),
        )
        FINAL_REPORT_RESULT = run_cross_dataset_report_notebook(
            PROJECT_ROOT,
            model_name=MODEL_NAME,
            selected_runs={
                dataset_id: result["run_dir"]
                for dataset_id, result in EXECUTION_RESULTS.items()
            },
            include_survface_faithfulness=(
                cross_model["status"] == "completed"
            ),
            write_outputs=WRITE_FINAL_REPORT,
            overwrite_outputs=True,
            rfw_evaluation_dir=(
                RFW_RESULT["output_dir"]
                if RFW_RESULT.get("status") == "completed"
                else None
            ),
            cross_model_run_matrix=(
                CROSS_MODEL_RUN_MATRIX or None
            ),
        )
        FINAL_REPORT_RESULT["cross_model_faithfulness"] = cross_model
else:
    EXECUTION_RESULTS = {
        dataset_id: {
            "status": "not_started",
            "reason": "EXECUTE=False; plan과 preflight만 수행했습니다.",
        }
        for dataset_id in DATASET_IDS
    }

EXECUTION_RESULT = EXECUTION_RESULTS[DATASET_ID]  # 기존 단일-result 호환용 별칭
INTEGRATED_RESULT = {
    "execution": EXECUTION_RESULTS,
    "postprocessing": POSTPROCESS_RESULTS,
    "rfw_verification": RFW_RESULT,
    "final_report": FINAL_REPORT_RESULT,
}
pprint(INTEGRATED_RESULT, sort_dicts=False)


C:\ronbun\research\experiments\step4_workflow.py:971: DtypeWarning: Columns (0: is_mated) have mixed types. Specify dtype option on import or set low_memory=False.
  selected = pd.read_csv(selected_path)


[07:36:49] rfw_custom/full/ada | population Grad-CAM extraction | elapsed=14s | progress=10% processed=208 total=2048 rate=14.65/s eta=2m 06s
[07:36:51] rfw_custom/full/ada | population Grad-CAM extraction | elapsed=16s | progress=20% processed=412 total=2048 rate=26.24/s eta=1m 02s
[07:36:52] rfw_custom/full/ada | population Grad-CAM extraction | elapsed=17s | progress=30% processed=616 total=2048 rate=35.83/s eta=40s
[07:36:54] rfw_custom/full/ada | population Grad-CAM extraction | elapsed=19s | progress=40% processed=820 total=2048 rate=43.84/s eta=28s
[07:36:55] rfw_custom/full/ada | population Grad-CAM extraction | elapsed=20s | progress=50% processed=1024 total=2048 rate=50.62/s eta=20s
[07:36:57] rfw_custom/full/ada | population Grad-CAM extraction | elapsed=22s | progress=60% processed=1232 total=2048 rate=56.66/s eta=14s
[07:36:58] rfw_custom/full/ada | population Grad-CAM extraction | elapsed=23s | progress=70% processed=1436 total=2048 rate=61.79/s eta=10s
[07:37:00] rfw_cus

C:\ronbun\research\experiments\step4_workflow.py:1128: DtypeWarning: Columns (0: is_mated) have mixed types. Specify dtype option on import or set low_memory=False.
  selected = pd.read_csv(selected_path)
C:\ronbun\research\experiments\step4_workflow.py:1246: DtypeWarning: Columns (0: is_mated) have mixed types. Specify dtype option on import or set low_memory=False.
  selected = pd.read_csv(selected_path)


[07:37:18] rfw_custom/full/ada | rfw_custom compressor fit | elapsed=43s | progress=50% processed=5 total=10 rate=0.11/s eta=43s family=pca
[07:37:21] rfw_custom/full/ada | rfw_custom compressor fit | elapsed=46s | progress=60% processed=6 total=10 rate=0.13/s eta=31s family=pq profile=pq_512_m8_b8
[07:37:26] rfw_custom/full/ada | rfw_custom compressor fit | elapsed=51s | progress=70% processed=7 total=10 rate=0.14/s eta=22s family=pq profile=pq_512_m16_b8
[07:37:35] rfw_custom/full/ada | rfw_custom compressor fit | elapsed=1m 00s | progress=80% processed=8 total=10 rate=0.13/s eta=15s family=pq profile=pq_512_m32_b8
[07:37:55] rfw_custom/full/ada | rfw_custom compressor fit | elapsed=1m 20s | progress=90% processed=9 total=10 rate=0.11/s eta=9s family=pq profile=pq_512_m64_b8
[07:38:09] rfw_custom/full/ada | rfw_custom compressor fit | elapsed=1m 34s | progress=100% processed=10 total=10 rate=0.11/s eta=0s family=pq profile=pq_512_m128_b8
[07:38:17] rfw_custom/full/ada | rfw_custom co

C:\ronbun\research\experiments\step4_workflow.py:1905: DtypeWarning: Columns (0: is_mated) have mixed types. Specify dtype option on import or set low_memory=False.
  selected = pd.read_csv(selected_path)
C:\ronbun\scripts\refresh_step4_search_spaces.py:432: DtypeWarning: Columns (0: is_mated) have mixed types. Specify dtype option on import or set low_memory=False.
  selected = pd.read_csv(context["selected_path"])


{"event": "rfw_custom compressor fit", "family": "pca", "processed": 5, "total": 5}
{"event": "rfw_custom compression retrieval", "processed": 128, "profile": "pca_384", "representation": "origin", "split": "calibration", "total": 460760}
{"event": "rfw_custom compression retrieval", "processed": 46076, "profile": "pca_384", "representation": "compressed", "split": "evaluation", "total": 460760}
{"event": "rfw_custom compression retrieval", "processed": 92152, "profile": "pca_384", "representation": "compressed", "search_mode": "pca_reconstruction_cosine", "split": "evaluation", "total": 460760}
{"event": "rfw_custom compression retrieval", "processed": 138228, "profile": "pca_256", "representation": "compressed", "split": "evaluation", "total": 460760}
{"event": "rfw_custom compression retrieval", "processed": 184304, "profile": "pca_256", "representation": "compressed", "search_mode": "pca_reconstruction_cosine", "split": "evaluation", "total": 460760}
{"event": "rfw_custom compressi

C:\ronbun\scripts\refresh_step4_search_spaces.py:432: DtypeWarning: Columns (0: is_mated) have mixed types. Specify dtype option on import or set low_memory=False.
  selected = pd.read_csv(context["selected_path"])


{"event": "rfw_custom compressor fit", "family": "pq", "processed": 1, "profile": "pq_512_m8_b8", "total": 5}
{"event": "rfw_custom compressor fit", "family": "pq", "processed": 2, "profile": "pq_512_m16_b8", "total": 5}
{"event": "rfw_custom compressor fit", "family": "pq", "processed": 3, "profile": "pq_512_m32_b8", "total": 5}
{"event": "rfw_custom compressor fit", "family": "pq", "processed": 4, "profile": "pq_512_m64_b8", "total": 5}
{"event": "rfw_custom compressor fit", "family": "pq", "processed": 5, "profile": "pq_512_m128_b8", "total": 5}
{"event": "rfw_custom compression retrieval", "processed": 128, "profile": "pq_512_m8_b8", "representation": "origin", "split": "calibration", "total": 230380}
{"event": "rfw_custom compression retrieval", "processed": 23118, "profile": "pq_512_m8_b8", "representation": "origin", "split": "evaluation", "total": 230380}
{"event": "rfw_custom compression retrieval", "processed": 46076, "profile": "pq_512_m8_b8", "representation": "compressed",

## 7. 결과 해석 경계

- `quick`은 코드·artifact 흐름과 경향 확인용이며 논문 최종 수치가 아닙니다.
- `full`끼리도 model UID, preprocessing, protocol, calibration, codec lineage가 같을 때만 직접 비교합니다.
- ArcFace·AdaFace·MagFace·EdgeFace 비교 단위는 선택한 pretrained checkpoint입니다. loss 함수 자체의 인과적 우월성으로 해석하지 않습니다.
- RFW-Custom은 비공식 identity-disjoint 1:N DIR/FPIR이며 RFW-Official의 pair/fold를 사용하지 않습니다.
- EdgeFace를 포함한 checkpoint와 RFW identity overlap은 `UNKNOWN`입니다. RFW 결과를 strict unseen-identity evidence로 사용하지 않습니다.
- RFW-Official은 1:1 TAR/FAR/EER supplementary 결과이며 open-set 표와 결합하지 않습니다. RFW-Custom에서 fitted/calibrated한 항목을 Official에 적용하면 same-domain diagnostic일 뿐 strict external transfer가 아닙니다.
- CI는 probe-level Wilson 및 paired bootstrap입니다. identity-cluster 또는 checkpoint 재학습 불확실성을 뜻하지 않습니다.
- PQ reconstruction cosine과 exhaustive ADC는 별도 search mode입니다. ADC를 IVF-PQ 또는 pgvector ANN latency로 해석하지 않습니다.
- Grad-CAM faithfulness는 검증된 high/low/random control artifact가 있을 때만 포함합니다.
